# EvalForge — reproducible GPU experiment
Enable internet and a CUDA GPU. This notebook creates a Python 3.11 environment independently of Kaggle’s host Python. The data mirror is pinned and checksum verified; model and dataset terms apply. No secret tokens are required for the public model and data. Outputs contain the adapter, manifest, and paired evaluation report.


In [ ]:
import subprocess, sys, os
from pathlib import Path
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'venv', '--python', '3.11', '--seed', '/kaggle/temp/evalforge-env'], check=True)
repo = Path('/kaggle/working/EvalForge')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Navaneeth-007/EvalForge.git', str(repo)], check=True)
print(subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True))
os.environ['HF_HOME'] = '/kaggle/temp/evalforge-model-cache'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['WANDB_MODE'] = 'disabled'
subprocess.run(['uv', 'pip', 'install', '--python', '/kaggle/temp/evalforge-env/bin/python', '-r', str(repo / 'requirements-ml.txt')], check=True)


## Train and evaluate
The default run uses three epochs on the filtered Spider train split, followed by paired evaluation on 200 development examples. Batch size is 1 with gradient accumulation 16 on one GPU. Runtime depends on the assigned hardware. Keep the session active and preserve the output artifacts.


In [ ]:
subprocess.run(['/kaggle/temp/evalforge-env/bin/python', '-u', '-m', 'scripts.run_gpu'], cwd=str(repo), env=os.environ.copy(), check=True)


## Inspect and save evidence
These are development metrics, not a blind test score. Inspect incorrect predictions before claiming an improvement. Do not publish the model card until its pending fields have been replaced with measured values.


In [ ]:
import json, shutil
manifest = json.loads((repo / 'outputs/phi3-sql-qlora/training_manifest.json').read_text())
report = json.loads((repo / 'outputs/eval_results.json').read_text())
print('Training examples:', manifest['train_examples'])
print('GPU:', manifest['gpu'], 'Elapsed seconds:', manifest['elapsed_seconds'])
print('Base metrics:', report['base_model_metrics'])
print('Adapter metrics:', report['finetuned_model_metrics'])
shutil.make_archive('/kaggle/working/evalforge-evidence', 'zip', repo / 'outputs')
